<a href="https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/colab-noui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<p align="center">
  
# Advanced RVC Inference — CLI Only

---

<div align="center">

Full command-line interface for RVC voice conversion, audio separation, training, and model management — no web UI required.

Ideal for headless environments, automation, and users who prefer the terminal.

---

[Discord](https://discord.gg/hvmsukmBHE) · [Github](https://github.com/ArkanDash/Advanced-RVC-Inference) · [CLI Guide](https://github.com/ArkanDash/Advanced-RVC-Inference/wiki/Cli-Guide) · [Main Colab](https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/Advanced-RVC.ipynb)

# ⚙️ Setup

Mount Google Drive and clone the repo locally. This notebook skips the Gradio web UI for a lighter, faster setup.

In [ ]:
# @title Mount Google Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Install Dependencies
from IPython.display import clear_output
import os

REPO_DIR = "/content/Advanced-RVC-Inference"
LOGS_PATH = f"{REPO_DIR}/advanced_rvc_inference/assets/logs"
BACKUPS_PATH = "/content/drive/MyDrive/RVCBackup"
CLI = f"{REPO_DIR}/rvc-cli.sh"

# Install system dependencies
!apt-get -y install libportaudio2 ffmpeg git -qq > /dev/null 2>&1

# Install uv for faster package management
!pip install uv -q

# Clone or update the repo
if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull origin master -q
else:
    !git clone https://github.com/ArkanDash/Advanced-RVC-Inference.git /content/Advanced-RVC-Inference -q
    %cd $REPO_DIR

# Install Python dependencies (skip gradio for headless mode)
!uv pip install -r requirements.txt --system -q 2>/dev/null || uv pip install -r requirements.txt --system -q

# Ensure the package itself is importable (critical for {REPO_DIR}/rvc-cli.sh)
%cd $REPO_DIR
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

clear_output()
print("✅ Advanced RVC Inference installed!")
print("📖 Full CLI guide: https://github.com/ArkanDash/Advanced-RVC-Inference/wiki/Cli-Guide")
print()
print("Run `{REPO_DIR}/rvc-cli.sh --help` to see all available commands.")

In [ ]:
# @title Set Environment Variables
# @markdown Tune these for better performance or compatibility.
import os
import sys

# Ensure working directory is always the repo root
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("✅ Environment variables set.")
print(f"  TF_CPP_MIN_LOG_LEVEL = {os.environ['TF_CPP_MIN_LOG_LEVEL']}")
print(f"  PYTORCH_ENABLE_MPS_FALLBACK = {os.environ['PYTORCH_ENABLE_MPS_FALLBACK']}")
print(f"  CUDA_LAUNCH_BLOCKING = {os.environ['CUDA_LAUNCH_BLOCKING']}")
print(f"  Working directory: {os.getcwd()}")

# 📋 System Info

Check your GPU, installed models, and available F0 methods.

In [ ]:
# @title System Info
import os, sys
os.chdir(REPO_DIR)
!{REPO_DIR}/rvc-cli.sh info

In [ ]:
# @title List Installed Models
import os
os.chdir(REPO_DIR)
!{REPO_DIR}/rvc-cli.sh list-models

In [ ]:
# @title List F0 Methods
import os
os.chdir(REPO_DIR)
!{REPO_DIR}/rvc-cli.sh list-f0-methods

# ⬇️ Download Models

Download RVC models from HuggingFace or other supported sources.

In [ ]:
# @title Download Model
# @markdown Paste a HuggingFace, Google Drive, MediaFire, or Mega URL.
download_url = ""  # @param {type:"string"}
model_name = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh download -l \"{download_url}\""
if model_name:
    cmd += f" -n \"{model_name}\""

!{cmd}

# 🎤 Voice Conversion (Inference)

Convert voice in audio files using an RVC model.

In [ ]:
# @title Voice Conversion
# @markdown Convert audio using an RVC model with full parameter control.
model_path = ""  # @param {type:"string"}
input_audio = ""  # @param {type:"string"}
output_audio = ""  # @param {type:"string"}
pitch_shift = 0  # @param {type:"integer"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin", "mangio-crepe-full", "hybrid"]
output_format = "wav"  # @param ["wav", "mp3", "flac", "ogg"]
filter_radius = 3  # @param {type:"integer"}
index_rate = 0.5  # @param {type:"number"}
rms_mix_rate = 1.0  # @param {type:"number"}
protect = 0.33  # @param {type:"number"}
split_audio = False  # @param {type:"boolean"}
f0_autotune = False  # @param {type:"boolean"}
formant_shifting = False  # @param {type:"boolean"}
clean_audio = False  # @param {type:"boolean"}
checkpointing = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh infer -m \"{model_path}\" -i \"{input_audio}\""
if output_audio:
    cmd += f" -o \"{output_audio}\""
cmd += f" -p {pitch_shift} --f0_method {f0_method} -f {output_format}"
cmd += f" --filter_radius {filter_radius} --index_rate {index_rate}"
cmd += f" --rms_mix_rate {rms_mix_rate} --protect {protect}"
if split_audio:
    cmd += " --split_audio"
if f0_autotune:
    cmd += " --f0_autotune"
if formant_shifting:
    cmd += " --formant_shifting"
if clean_audio:
    cmd += " --clean_audio"
if checkpointing:
    cmd += " --checkpointing"

!{cmd}

# 🎵 Audio Separation (UVR)

Separate vocals from instrumentals using UVR5 models.

In [ ]:
# @title Audio Separation
# @markdown Separate vocals from instrumentals using UVR5 models.
input_audio = ""  # @param {type:"string"}
output_dir = ""  # @param {type:"string"}
uvr_model = "MDXNET_Main"  # @param ["MDXNET_Main", "MDX-NET-2", "MDX-Vocals-2", "BS-Roformer", "Roformer", "MDX-Version-1"]
output_format = "wav"  # @param ["wav", "mp3", "flac"]
aggression = 5  # @param {type:"integer"}
enable_tta = False  # @param {type:"boolean"}
enable_denoise = False  # @param {type:"boolean"}
separate_backing = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh uvr -i \"{input_audio}\" --model {uvr_model} -f {output_format}"
cmd += f" --aggression {aggression}"
if output_dir:
    cmd += f" -o \"{output_dir}\""
if enable_tta:
    cmd += " --enable_tta"
if enable_denoise:
    cmd += " --enable_denoise"
if separate_backing:
    cmd += " --separate_backing"

!{cmd}

# 🏋️ Training Pipeline

Full training pipeline: create dataset → extract features → preprocess → create index → train.

Run each step in order.

In [ ]:
# @title Step 1: Create Dataset
# @markdown Build a training dataset from YouTube URLs or local audio files. Provide a comma-separated list of YouTube URLs or a local directory path.
source = ""  # @param {type:"string"}
output_dir = "./dataset"  # @param {type:"string"}
sample_rate = 48000  # @param ["40000", "48000"]
clean_dataset = True  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"number"}
separate_vocals = True  # @param {type:"boolean"}
skip_start = 0  # @param {type:"integer"}
skip_end = 0  # @param {type:"integer"}

import os
os.chdir(REPO_DIR)

# Auto-detect if source is a URL or local path
if source.startswith("http"):
    cmd = f"{REPO_DIR}/rvc-cli.sh create-dataset -u \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"
else:
    cmd = f"{REPO_DIR}/rvc-cli.sh create-dataset -i \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"

if clean_dataset:
    cmd += f" --clean_dataset --clean_strength {clean_strength}"
if not separate_vocals:
    cmd += " --no-separate"
if skip_start > 0:
    cmd += f" --skip_start {skip_start}"
if skip_end > 0:
    cmd += f" --skip_end {skip_end}"

!{cmd}

In [ ]:
# @title Step 2: Preprocess
# @markdown Slice and normalize training audio data.
model_name = ""  # @param {type:"string"}
sample_rate = 48000  # @param ["40000", "48000"]
dataset_path = "./dataset"  # @param {type:"string"}
cpu_cores = 2  # @param {type:"integer"}
cut_method = "Automatic"  # @param ["Automatic", "Simple", "Skip"]
clean_dataset = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh preprocess {model_name} --sample_rate {sample_rate}"
cmd += f" --dataset_path \"{dataset_path}\" --cpu_cores {cpu_cores}"
cmd += f" --cut_method {cut_method}"
if clean_dataset:
    cmd += " --clean_dataset"

!{cmd}

In [ ]:
# @title Step 3: Extract Features
# @markdown Extract embeddings and F0 features from preprocessed data.
model_name = ""  # @param {type:"string"}
sample_rate = 48000  # @param ["40000", "48000"]
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin"]
rvc_version = "v2"  # @param ["v1", "v2"]
cpu_cores = 2  # @param {type:"integer"}
gpu_id = "-"  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh extract {model_name} --sample_rate {sample_rate}"
cmd += f" --f0_method {f0_method} --version {rvc_version}"
cmd += f" --cpu_cores {cpu_cores} --gpu {gpu_id}"

!{cmd}

In [ ]:
# @title Step 4: Create Index
# @markdown Create the .index file for voice retrieval.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
algorithm = "Auto"  # @param ["Auto", "Faiss", "KMeans"]

import os
os.chdir(REPO_DIR)

!{REPO_DIR}/rvc-cli.sh create-index {model_name} --version {rvc_version} --algorithm {algorithm}

In [ ]:
# @title Step 5: Train Model
# @markdown Train the RVC voice model. This may take a while depending on epochs and data size.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
sample_rate = 48000  # @param ["40000", "48000"]
epochs = 300  # @param {type:"integer"}
batch_size = 8  # @param ["4", "8", "16", "32"]
save_every = 50  # @param {type:"integer"}
gpu_id = "0"  # @param {type:"string"}
author = ""  # @param {type:"string"}
optimizer = "AdamW"  # @param ["AdamW", "ScheduleFreeAdamW", "Muon", "Sophia", "Lion", "Prodigy", "NAdam", "RAdam", "Adan", "AnyPrecisionAdamW", "Ranger21", "AdaFactor", "DAdaptAdam", "Adam", "PAdam", "Apollo", "CAME", "NovoGrad", "ScheduleFreeAdam", "DAdaptAdaGrad", "SGD", "RMSprop", "AdaBelief", "AdaBeliefV2", "LAMB", "LARS", "Adagrad", "Adadelta", "Adamax", "ASGD", "DAdaptSGD", "QHAdam", "SWATS", "Shampoo", "SOAP", "A2Grad", "AggMo", "PID", "Yogi", "Fromage", "SM3", "ScheduleFreeSGD", "Nero"]
overtrain_detect = False  # @param {type:"boolean"}
pitch_guidance = True  # @param {type:"boolean"}
cache_gpu = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh train {model_name} --version {rvc_version}"
cmd += f" --epochs {epochs} --batch_size {batch_size}"
cmd += f" --save_every {save_every} --gpu {gpu_id}"
cmd += f" --optimizer {optimizer}"
if author:
    cmd += f" --author \"{author}\""
if overtrain_detect:
    cmd += " --overtrain_detect"
if not pitch_guidance:
    cmd += " --no-pitch_guidance"
if cache_gpu:
    cmd += " --cache_gpu"

!{cmd}

In [ ]:
# @title Create Reference Set
# @markdown Create a reference audio set for improved inference quality.
audio_file = ""  # @param {type:"string"}
ref_name = "reference"  # @param {type:"string"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "fcpe", "harvest"]
pitch_shift = 0  # @param {type:"integer"}

import os
os.chdir(REPO_DIR)

cmd = f"{REPO_DIR}/rvc-cli.sh create-ref \"{audio_file}\" -n {ref_name}"
cmd += f" --f0_method {f0_method}"
if pitch_shift != 0:
    cmd += f" --pitch_shift {pitch_shift}"

!{cmd}

# 💾 Backup & Restore

Backup models to Google Drive and restore them later.

In [ ]:
# @title Backup Model to Drive
# @markdown Copy a trained model from Colab to your Google Drive for safekeeping.
model_name = "" # @param {type:"string"}

import os, shutil
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  pass

src = f"{LOGS_PATH}/{model_name}"
dst = f"{BACKUPS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Model '{model_name}' not found at {src}")
else:
    os.makedirs(BACKUPS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Backed up '{model_name}' to Google Drive.")

In [ ]:
# @title List Backed Up Models

import os

if not os.path.exists(BACKUPS_PATH):
    print(f"❌ Backup directory not found: {BACKUPS_PATH}")
else:
    models = [d for d in os.listdir(BACKUPS_PATH) if os.path.isdir(os.path.join(BACKUPS_PATH, d))]
    if not models:
        print("No backed up models found.")
    else:
        for i, m in enumerate(models):
            size = sum(os.path.getsize(os.path.join(BACKUPS_PATH, m, f)) for f in os.listdir(os.path.join(BACKUPS_PATH, m)) if os.path.isfile(os.path.join(BACKUPS_PATH, m, f)))
            size_mb = size / (1024*1024)
            print(f"  {i+1}. {m} ({size_mb:.1f} MB)")

In [ ]:
# @title Restore Model from Drive
# @markdown Load a previously backed up model from Google Drive into Colab.
model_name = "" # @param {type:"string"}

import os, shutil

src = f"{BACKUPS_PATH}/{model_name}"
dst = f"{LOGS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Backup '{model_name}' not found in Google Drive.")
else:
    os.makedirs(LOGS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Restored '{model_name}' from Google Drive.")

# 📤 Push to HuggingFace

Upload your trained model to HuggingFace to share it with others.

In [ ]:
# @title Push Model to HuggingFace
# @markdown Upload your trained model. Get a token from https://huggingface.co/settings/tokens
hf_token = "" #@param {type:"string"}
repo_id = "" #@param {type:"string"}
model_name = "" #@param {type:"string"}

import os, shutil
from huggingface_hub import HfApi, create_repo, login

login(hf_token)
api = HfApi()

try:
    create_repo(repo_id)
except Exception as e:
    print(f"Note: {e}. Proceeding...")

model_path = f"{LOGS_PATH}/{model_name}"
zip_path = f"{model_path}/{model_name}.zip"

if not os.path.exists(zip_path):
    print(f"Zipping {model_path}...")
    shutil.make_archive(f"{model_path}/{model_name}", 'zip', model_path)
    zip_path = f"{model_path}/{model_name}.zip"

api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo=f"{model_name}.zip",
    repo_id=repo_id,
    repo_type="model"
)

print(f"\n✅ Model uploaded to https://huggingface.co/{repo_id}")

# 📖 CLI Quick Reference

| Command | Description |
|---------|-------------|
| `{REPO_DIR}/rvc-cli.sh --help` | Show all commands |
| `{REPO_DIR}/rvc-cli.sh infer --help` | Inference options |
| `{REPO_DIR}/rvc-cli.sh uvr --help` | Audio separation options |
| `{REPO_DIR}/rvc-cli.sh train --help` | Training options |
| `{REPO_DIR}/rvc-cli.sh create-dataset --help` | Dataset creation |
| `{REPO_DIR}/rvc-cli.sh extract --help` | Feature extraction |
| `{REPO_DIR}/rvc-cli.sh preprocess --help` | Preprocessing |
| `{REPO_DIR}/rvc-cli.sh create-index --help` | Index creation |
| `{REPO_DIR}/rvc-cli.sh create-ref --help` | Reference creation |
| `{REPO_DIR}/rvc-cli.sh download --help` | Model download |
| `{REPO_DIR}/rvc-cli.sh info` | System info |
| `{REPO_DIR}/rvc-cli.sh list-models` | List models |
| `{REPO_DIR}/rvc-cli.sh list-f0-methods` | List F0 methods |
| `{REPO_DIR}/rvc-cli.sh version` | Show version |